# LGEO2270 | Regional hydrological preprocessing
**Teacher-only notebook · UCLouvain Earth and Life Institute / School of Geography**

Create a single four-band GeoTIFF for a public Earth Engine image asset. Run this once for the region, not once per student.

The processing follows the student notebook: Copernicus GLO-30 resampled to an aligned 20 m Belgian Lambert 72 grid → RHW river rasterization → SAGA stream burning (method 0, epsilon 1 m) → sink filling (minimum slope 0.01°) → D8 drainage → upstream cell counts and boundary provenance.

The default extent is an illustrative Condroz–Famenne rectangle, not an official excursion boundary. Modify the coordinates as needed. The extra margin is computational context and is retained in the exported asset so that upstream traversal remains possible. All hydrology is calculated across the entire rectangle in one pass, not independent tiles.

| Band | Definition |
|---|---|
| dem_conditioned | metres |
| flow_direction | ESRI D8: E=1, SE=2, S=4, SW=8, W=16, NW=32, N=64, NE=128; 0=no internal successor |
| flow_accumulation | upstream cells including self; multiply by 0.04 for hectares |
| edge_connected | 1 if any upstream cell touches the processing rectangle, otherwise 0 |

The edge flag is a conservative coverage diagnostic, not proof of true hydrological accuracy. RHW is a Walloon dataset; drainage outside its mapped coverage is conditioned from elevation alone. The output is a 20 m resampling of a roughly 30 m DEM, not a 20 m measurement.

[Model workflow](https://github.com/kvanoost/Watem/blob/main/assets/model_workflow.svg)

In [ ]:
#@title 1 — Define study region and install preprocessing software
WEST=4.70 #@param {type:"number"}
SOUTH=50.00 #@param {type:"number"}
EAST=5.65 #@param {type:"number"}
NORTH=50.55 #@param {type:"number"}
MARGIN_KM=8 #@param {type:"number"}
ASSET_NAME="Condroz_Hydrology_20m_v1" #@param {type:"string"}
VERBOSE=False #@param {type:"boolean"}
RES=20
CRS="EPSG:31370"
BURN_DEPTH=1.0
MIN_SLOPE=0.01
MAX_CELLS=25_000_000
from pathlib import Path
import os, sys, math, shutil, subprocess, tempfile, time, re, json, hashlib
from datetime import datetime, timezone
assert -180<WEST<EAST<180 and -90<SOUTH<NORTH<90
assert 0<=MARGIN_KM<=20 and re.fullmatch(r"[A-Za-z0-9_-]+",ASSET_NAME)
WORK=Path(tempfile.mkdtemp(prefix="watem_regional_",dir="/content"))
def run(args):
    p=subprocess.run(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if VERBOSE or p.returncode: print(p.stdout[-6000:],flush=True)
    p.check_returncode()
    return p.stdout
t0=time.perf_counter()
if not shutil.which("saga_cmd"):
    run(["apt-get","update","-qq"])
    run(["apt-get","install","-y","saga"])
# Only standard SAGA terrain tools are required; no WaTEM binary, wrapper or Git LFS.
run([sys.executable,"-m","pip","install","-q","pysheds","numba","geopandas","rasterio","requests"])
import numpy as np
import geopandas as gpd
import rasterio
import requests
import matplotlib.pyplot as plt
from rasterio.warp import transform_bounds,reproject,Resampling
from rasterio.transform import from_origin
from rasterio.features import rasterize
from shapely.geometry import box
from numba import njit
from pysheds.grid import Grid
from IPython.display import display
from pyproj import Transformer
bb=transform_bounds("EPSG:4326",CRS,WEST,SOUTH,EAST,NORTH,densify_pts=21)
margin=math.ceil(MARGIN_KM*1000/RES)*RES
xmin,ymin=np.floor(np.array(bb[:2])/RES)*RES-margin
xmax,ymax=np.ceil(np.array(bb[2:])/RES)*RES+margin
width,height=int((xmax-xmin)/RES),int((ymax-ymin)/RES)
assert width*height<=MAX_CELLS, f"{width*height:,} cells exceeds the conservative Colab memory limit. Reduce region or use a high-memory runtime and deliberately revise MAX_CELLS."
tr=from_origin(xmin,ymax,RES,RES)
profile=dict(driver="GTiff",height=height,width=width,count=1,crs=CRS,transform=tr,nodata=-9999,compress="deflate",tiled=True,blockxsize=256,blockysize=256)
print(f"Grid: {width:,} × {height:,} = {width*height:,} cells; extent {width*RES/1000:.1f} × {height*RES/1000:.1f} km")
print(f"Uncompressed four-band Float32 output: {width*height*16/1e6:.0f} MB")
print("Software and grid ready.")
# Diagnostic geographic coverage map: study rectangle and full processing rectangle.
import folium
m=folium.Map(location=[(SOUTH+NORTH)/2,(WEST+EAST)/2],zoom_start=9)
folium.Rectangle([[SOUTH,WEST],[NORTH,EAST]],color="green",fill=False,tooltip="Study extent").add_to(m)
inv=Transformer.from_crs(CRS,4326,always_xy=True)
corners=[inv.transform(x,y) for x,y in [(xmin,ymin),(xmax,ymin),(xmax,ymax),(xmin,ymax),(xmin,ymin)]]
folium.PolyLine([(y,x) for x,y in corners],color="orange",tooltip="Full processing/export extent").add_to(m)
m.fit_bounds([(y,x) for x,y in corners])
display(m)

In [ ]:
#@title 2 — Retrieve DEM and official RHW streams
t=time.perf_counter()
west,south,east,north=transform_bounds(CRS,"EPSG:4326",xmin,ymin,xmax,ymax,densify_pts=21)
dem_sources=[]
for lon in range(math.floor(west),math.floor(east)+1):
    for lat in range(math.floor(south),math.floor(north)+1):
        tile=f"Copernicus_DSM_COG_10_N{lat:02d}_00_E{lon:03d}_00_DEM"
        dem_sources.append(f"https://copernicus-dem-30m.s3.amazonaws.com/{tile}/{tile}.tif")
dem=np.full((height,width),-9999,dtype="float32")
for url in dem_sources:
    print("Reading DEM tile",url.split("/")[-2],flush=True)
    tmp=np.full_like(dem,-9999)
    with rasterio.Env(GDAL_HTTP_TIMEOUT="120",GDAL_HTTP_MAX_RETRY="3"):
        with rasterio.open(url) as ds:
            reproject(rasterio.band(ds,1),tmp,dst_transform=tr,dst_crs=CRS,dst_nodata=-9999,resampling=Resampling.bilinear)
    valid=tmp!=-9999
    dem[valid]=tmp[valid]
    del tmp,valid
assert np.isfinite(dem).all() and (dem!=-9999).all(),"Incomplete DEM coverage"
def write_single(path,array,dtype):
    with rasterio.open(path,"w",**{**profile,"dtype":dtype}) as ds: ds.write(array.astype(dtype,copy=False),1)
write_single(WORK/"dem.tif",dem,"float32")
RIVER_URL="https://geoservices.wallonie.be/arcgis/rest/services/EAU/RES_HYDRO_WAL/MapServer/1"
session=requests.Session()
def query(url,params,post=False):
    for attempt in range(3):
        try:
            res=session.post(url,data=params,timeout=120) if post else session.get(url,params=params,timeout=120)
            res.raise_for_status()
            data=res.json()
            if "error" in data: raise RuntimeError(data["error"])
            return data
        except (requests.RequestException,ValueError,RuntimeError):
            if attempt==2: raise
            time.sleep(2**attempt)
meta=query(RIVER_URL,{"f":"json"})
assert meta["geometryType"]=="esriGeometryPolyline"
ids=query(RIVER_URL+"/query",dict(where="1=1",geometry=",".join(map(str,(xmin,ymin,xmax,ymax))),geometryType="esriGeometryEnvelope",inSR=31370,spatialRel="esriSpatialRelIntersects",returnIdsOnly="true",f="json")).get("objectIds") or []
assert ids,"No RHW streams returned"
ids=sorted(set(ids))
print(f"Downloading {len(ids):,} RHW features",flush=True)
features=[]
batch_size=min(200,int(meta.get("maxRecordCount") or 200))
for i in range(0,len(ids),batch_size):
    part=ids[i:i+batch_size]
    j=query(RIVER_URL+"/query",dict(objectIds=",".join(map(str,part)),outFields="OBJECTID",outSR=4326,returnGeometry="true",f="geojson"),post=True)
    assert not j.get("exceededTransferLimit") and len(j["features"])==len(part),"Truncated RHW batch"
    features.extend(j["features"])
    if i%(batch_size*20)==0: print(f"  {len(features):,}/{len(ids):,}",flush=True)
g=gpd.GeoDataFrame.from_features(features,crs=4326).to_crs(CRS)
del features
assert len(g)==len(ids)
g.geometry=g.geometry.make_valid()
g=gpd.clip(g,box(xmin,ymin,xmax,ymax),keep_geom_type=True).explode(index_parts=False)
g=g[g.geometry.geom_type=="LineString"].copy()
assert len(g)
river_cells=rasterize(((geom,1) for geom in g.geometry),out_shape=dem.shape,transform=tr,fill=0,all_touched=True,dtype="uint8")
write_single(WORK/"stream_mask.tif",np.where(river_cells==1,1,-9999),"int16")
print(f"DEM and {len(g):,} clipped stream lines prepared in {time.perf_counter()-t:.1f} s")

In [ ]:
#@title 3 — River burning and depression filling
t=time.perf_counter()
def saga(*args):
    return run(["saga_cmd",*args])
# Clean stale files if this block is repeated.
for stem in ("original","streams","burned","hydro","hydro_fdir","hydro_wshed"):
    for ext in (".sgrd",".sdat",".mgrd",".prj",".sdat.aux.xml"):
        (WORK/(stem+ext)).unlink(missing_ok=True)
(WORK/"hydro.tif").unlink(missing_ok=True)
saga("io_gdal",0,"-FILES",WORK/"dem.tif","-GRIDS",WORK/"original.sgrd")
saga("io_gdal",0,"-FILES",WORK/"stream_mask.tif","-GRIDS",WORK/"streams.sgrd")
print("Burning mapped streams...",flush=True)
saga("ta_preprocessor",6,"-DEM",WORK/"original.sgrd","-STREAM",WORK/"streams.sgrd","-BURN",WORK/"burned.sgrd","-METHOD",0,"-EPSILON",BURN_DEPTH)
print("Filling depressions across entire region...",flush=True)
saga("ta_preprocessor",4,"-ELEV",WORK/"burned.sgrd","-FILLED",WORK/"hydro.sgrd","-FDIR",WORK/"hydro_fdir.sgrd","-WSHED",WORK/"hydro_wshed.sgrd","-MINSLOPE",MIN_SLOPE)
saga("io_gdal",2,"-GRIDS",WORK/"hydro.sgrd","-FILE",WORK/"hydro.tif")
with rasterio.open(WORK/"hydro.tif") as ds:
    assert ds.shape==dem.shape and ds.transform.almost_equals(tr) and ds.crs==rasterio.crs.CRS.from_string(CRS)
    hydro=ds.read(1).astype("float32")
assert np.isfinite(hydro).all() and (hydro!=-9999).all()
delta=hydro-dem
print(f"Conditioned elevation change min/median/max: {delta.min():.2f} / {np.median(delta):.2f} / {delta.max():.2f} m")
del delta
print(f"Terrain conditioning complete in {time.perf_counter()-t:.1f} s")

In [ ]:
#@title 4 — Calculate D8, accumulation and incomplete-basin flags
t=time.perf_counter()
if not hasattr(np,"in1d"):
    np.in1d=lambda a,b,assume_unique=False,invert=False: np.isin(np.asarray(a).ravel(),np.asarray(b).ravel(),assume_unique=assume_unique,invert=invert)
grid=Grid.from_raster(str(WORK/"hydro.tif"))
z=grid.read_raster(str(WORK/"hydro.tif"))
raw=np.asarray(grid.flowdir(z,dirmap=(64,128,1,2,4,8,16,32),routing="d8"))
codes=np.array([1,2,4,8,16,32,64,128])
fd=np.where(np.isin(raw,codes),raw,0).astype("uint8")
del raw,z,grid
# Single graph, topological traversal. Counts include each source cell.
# Boundary cells (including sinks) propagate their flag through every successor.
@njit
def graph(fd,hydro):
    h,w=fd.shape
    n=h*w
    dest=np.full(n,-1,np.int32)
    indegree=np.zeros(n,np.uint8)
    acc=np.ones(n,np.int32)
    edge=np.zeros(n,np.uint8)
    queue=np.empty(n,np.int32)
    codes=(64,128,1,2,4,8,16,32)
    drs=(-1,-1,0,1,1,1,0,-1)
    dcs=(0,1,1,1,0,-1,-1,-1)
    for r in range(h):
        for c in range(w):
            i=r*w+c
            if r==0 or c==0 or r==h-1 or c==w-1: edge[i]=1
            for k in range(8):
                if fd[r,c]==codes[k]:
                    a,b=r+drs[k],c+dcs[k]
                    if 0<=a<h and 0<=b<w:
                        if hydro[a,b]>=hydro[r,c]: raise ValueError("Non-downhill D8 link")
                        j=a*w+b
                        dest[i]=j
                        indegree[j]+=1
                    else:
                        fd[r,c]=0
                    break
            if dest[i]<0 and 0<r<h-1 and 0<c<w-1:
                raise ValueError("Interior cell has no downstream successor")
    tail=0
    for i in range(n):
        if indegree[i]==0:
            queue[tail]=i
            tail+=1
    head=0
    while head<tail:
        i=queue[head]
        head+=1
        j=dest[i]
        if j>=0:
            acc[j]+=acc[i]
            if edge[i]: edge[j]=1
            indegree[j]-=1
            if indegree[j]==0:
                queue[tail]=j
                tail+=1
    if tail!=n: raise ValueError("D8 graph contains a cycle")
    terminal_sum=np.int64(0)
    for i in range(n):
        if dest[i]<0: terminal_sum+=acc[i]
    if terminal_sum!=n: raise ValueError("Flow accumulation mass balance failed")
    return acc.reshape((h,w)),edge.reshape((h,w))
acc,edge=graph(fd,hydro)
# Verify actual accumulation fits Float32 exactly, including all integer cell counts.
# Use Float64 if necessary, rather than rounding any D8 counts.
out_dtype="float32" if int(acc.max())<=2**24 else "float64"
print(f"Graph checks passed: {acc.size:,} cells, largest contributing area {int(acc.max())*0.04:,.0f} ha")
print(f"Edge-connected cells: {100*edge.mean():.1f}% | output type: {out_dtype} | elapsed {time.perf_counter()-t:.1f} s")
# Independently traverse several interior complete outlet candidates.
from collections import deque
offsets={64:(-1,0),128:(-1,1),1:(0,1),2:(1,1),4:(1,0),8:(1,-1),16:(0,-1),32:(-1,-1)}
candidates=np.flatnonzero((edge==0)&(acc>=1250)&(acc<=125000)&(fd>0))
assert len(candidates),"No complete classroom catchment candidates; expand region"
rng=np.random.default_rng(2270)
for i in rng.choice(candidates,size=min(5,len(candidates)),replace=False):
    r,c=divmod(int(i),width)
    visited={(r,c)}; queue=deque(visited)
    while queue:
        a,b=queue.popleft()
        for k,(dr,dc) in offsets.items():
            y,x=a-dr,b-dc
            if 0<=y<height and 0<=x<width and fd[y,x]==k and (y,x) not in visited:
                visited.add((y,x)); queue.append((y,x))
    assert len(visited)==int(acc[r,c])
    assert not any(y in (0,height-1) or x in (0,width-1) for y,x in visited)
print("Independent reverse watershed checks passed.")
# Visual overview; max aggregation preserves thin drainage lines in regional preview.
from matplotlib.colors import ListedColormap, BoundaryNorm
stride=max(1,math.ceil(max(height,width)/1800))
categories=np.zeros(acc.shape,dtype="uint8")
categories[(acc>=1250)&(acc<=25000)&(edge==0)]=1
categories[(acc>25000)&(acc<=125000)&(edge==0)]=2
categories[(acc>=1250)&(acc<=125000)&(edge==1)]=3
categories[acc>125000]=4
# Pool each category independently to keep streams visible when downsampled.
from scipy.ndimage import maximum_filter
overview=maximum_filter(categories,size=stride)[::stride,::stride]
cmap=ListedColormap(["#f5f5f2","#d4aa00","#f79422","#9ca3af","#d93932"])
fig,ax=plt.subplots(1,2,figsize=(15,7),constrained_layout=True)
ax[0].imshow(hydro[::stride,::stride],cmap="terrain");ax[0].set_title("Conditioned DEM")
im=ax[1].imshow(overview,cmap=cmap,vmin=-.5,vmax=4.5);ax[1].set_title("Outlet suitability (regional overview)")
cb=fig.colorbar(im,ax=ax[1],ticks=range(5),shrink=.65)
cb.ax.set_yticklabels(["<50 ha","50–1000 ha","1000–5000 ha","Incomplete","Too large"])
for a in ax:a.axis("off")
plt.show()
del candidates,categories,overview

In [ ]:
#@title 5 — Export and download a single multiband GeoTIFF
from rasterio.shutil import copy as rio_copy
from google.colab import files
t=time.perf_counter()
bands=[("dem_conditioned",hydro),("flow_direction",fd),("flow_accumulation",acc),("edge_connected",edge)]
raw_tif=WORK/"stack.tif"
output=WORK/f"{ASSET_NAME}.tif"
output.unlink(missing_ok=True)
provenance=dict(schema_version="1",created_utc=datetime.now(timezone.utc).isoformat(),crs=CRS,resolution_m=RES,study_extent_wgs84=[WEST,SOUTH,EAST,NORTH],margin_km=MARGIN_KM,transform=list(tr)[:6],dem_sources=dem_sources,rhw_source=RIVER_URL,rhw_features=int(len(ids)),burn_method="SAGA ta_preprocessor 6 method 0",burn_epsilon_m=BURN_DEPTH,fill_method="SAGA ta_preprocessor 4",minimum_slope_deg=MIN_SLOPE,d8_encoding="E1 SE2 S4 SW8 W16 NW32 N64 NE128; 0 terminal",accumulation="cells including self",edge_flag="1 upstream path intersects raster perimeter; 0 otherwise")
with rasterio.open(raw_tif,"w",**{**profile,"count":4,"dtype":out_dtype,"BIGTIFF":"IF_SAFER","predictor":3}) as ds:
    for i,(name,array) in enumerate(bands,1):
        ds.set_band_description(i,name)
        for _,win in ds.block_windows(i):
            r,c=int(win.row_off),int(win.col_off)
            ds.write(array[r:r+int(win.height),c:c+int(win.width)].astype(out_dtype),i,window=win)
    ds.update_tags(**{k:json.dumps(v) if isinstance(v,(dict,list)) else str(v) for k,v in provenance.items()})
# COG layout, nearest-neighbor overviews only; no averaging of direction or edge flags.
rio_copy(raw_tif,output,driver="COG",COMPRESS="DEFLATE",PREDICTOR="FLOATING_POINT",BLOCKSIZE=512,OVERVIEWS="IGNORE_EXISTING",RESAMPLING="NEAREST",BIGTIFF="IF_SAFER")
with rasterio.open(output) as ds:
    assert ds.count==4 and ds.descriptions==tuple(name for name,_ in bands)
    assert ds.transform==tr and ds.crs==rasterio.crs.CRS.from_string(CRS)
    for i,(name,array) in enumerate(bands,1):
        for _,win in ds.block_windows(i):
            r,c=int(win.row_off),int(win.col_off)
            assert np.array_equal(ds.read(i,window=win),array[r:r+int(win.height),c:c+int(win.width)]),f"Readback mismatch: {name}"
raw_tif.unlink()
print(f"Verified: {output.name} | {output.stat().st_size/1e6:.1f} MB | {time.perf_counter()-t:.1f} s")
print("Upload this one file to Earth Engine. Use Sample pyramiding, and retain the original 20 m grid for numerical analysis.")
files.download(str(output))

## Upload to Google Earth Engine

1. Open the [Earth Engine Code Editor](https://code.earthengine.google.com/), Assets → NEW → Image upload.
2. Select the single exported .tif and set the destination asset ID.
3. In Advanced options select **Sample** pyramiding policy (do not average D8 directions or edge flags). If specifying no-data use -9999, not zero.
4. After ingestion check the band names and native projection. The GeoTIFF contains band descriptions, but the importer may assign generic names such as b1–b4. Rename on loading by positional order if necessary.
5. Verify redistribution permissions for Copernicus GLO-30 and RHW before enabling public read access.

[Official Earth Engine raster upload documentation](https://developers.google.com/earth-engine/guides/image_upload)

Example Code Editor JavaScript (replace asset ID):

```javascript
var hydro = ee.Image('projects/YOUR_PROJECT/assets/Condroz_Hydrology_20m_v1')
  .rename(['dem_conditioned','flow_direction','flow_accumulation','edge_connected']);
print(hydro.bandNames(), hydro.projection());
var ha = hydro.select('flow_accumulation').multiply(0.04);
var incomplete = hydro.select('edge_connected').eq(1);
var classes = ha.multiply(0).toByte()
  .where(ha.gte(50).and(ha.lte(1000)).and(incomplete.not()), 1)
  .where(ha.gt(1000).and(ha.lte(5000)).and(incomplete.not()), 2)
  .where(ha.gte(50).and(ha.lte(5000)).and(incomplete), 3)
  .where(ha.gt(5000), 4);
Map.centerObject(hydro, 9);
Map.addLayer(classes.selfMask(), {min:1,max:4,palette:['ffdf00','ff9b27','9ca3af','ef4444']}, 'Outlet suitability');
```

For numerical extraction use the asset's original projection and affine transform, not a resampled geographic grid or a coarse map pyramid. No actual Earth Engine upload is performed by this notebook; it creates the file for manual ingestion. The student notebook is not yet connected to this asset.